In [2]:
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.graph import StateGraph, MessagesState
from langgraph.graph import START, END
from llm_factory import LLMFactory
from langchain_core.messages import SystemMessage, trim_messages
from typing import TypedDict  

In [4]:
from config import DATA_DIR
files = [
   
    DATA_DIR / 'sh_arp_cache_2020-11-10074812.log',
    # DATA_DIR / 'Microsoft365DefenderEvents.json',
    # DATA_DIR / 'WindowsEvents.json',
    #DATA_DIR / "file_01.json",
    #DATA_DIR / "file_03.csv",
]

print(files)

with open(files[0], 'r') as f:
    data = f.read()
print(type(data))
print(f"Total de eventos: {len(data)}")
print(f"Exemplo:\n {data}")

[WindowsPath('C:/davi_tonon/mestrado/data/raw/sh_arp_cache_2020-11-10074812.log')]
<class 'str'>
Total de eventos: 1884
Exemplo:
 type=SYSCALL msg=audit(1604994496.155:92733): arch=c000003e syscall=59 success=yes exit=0 a0=558e251634a0 a1=558e25162a50 a2=558e25160800 a3=8 items=2 ppid=29002 pid=1631 auid=1000 uid=1000 gid=1000 euid=1000 suid=1000 fsuid=1000 egid=1000 sgid=1000 fsgid=1000 tty=pts0 ses=104 comm="arp" exe="/usr/sbin/arp" key=(null)
type=EXECVE msg=audit(1604994496.155:92733): argc=2 a0="arp" a1="-a"
type=CWD msg=audit(1604994496.155:92733): cwd="/home/wardog"
type=PATH msg=audit(1604994496.155:92733): item=0 name="/usr/sbin/arp" inode=13181 dev=08:01 mode=0100755 ouid=0 ogid=0 rdev=00:00 nametype=NORMAL cap_fp=0 cap_fi=0 cap_fe=0 cap_fver=0 cap_frootid=0
type=PATH msg=audit(1604994496.155:92733): item=1 name="/lib64/ld-linux-x86-64.so.2" inode=29514 dev=08:01 mode=0100755 ouid=0 ogid=0 rdev=00:00 nametype=NORMAL cap_fp=0 cap_fi=0 cap_fe=0 cap_fver=0 cap_frootid=0
type=PRO

In [6]:
SYSTEM_PROMPT = """
You are a cybersecurity analyst specialized in log analysis and MITRE ATT&CK mapping.
You will receive many event of a file. Analyze each event carefully.

You are able to:
- Detect anomalies in logs
- Identify security events
- Map findings to MITRE ATT&CK TTPs

Rules:
- Always base your analysis on evidence
- Do NOT hallucinate
- Be precise and technical
- Always answer in English

### OUTPUT REQUIREMENTS
Provide the results in the following structured format:

**MITRE ATT&CK Mapping**
- **Tactics:**
- **Techniques:**
- **Technique IDs:**



"""

In [7]:
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.graph import StateGraph, START, END
from langgraph.graph import MessagesState  # ← Importante!
from langchain_ollama import ChatOllama
from langchain_core.messages import HumanMessage

# ❌ NÃO use TypedDict customizado para messages
# ✅ Use MessagesState que já sabe acumular

def analyze_logs(state: MessagesState):
    llm = LLMFactory().get_model()
    #llm = ChatOllama(model="llama3.1:8b", temperature=0.1)
    
    # MessagesState já tem 'messages' como chave
    response = llm.invoke(state["messages"])
    
    # Retorna a nova mensagem (MessagesState acumula automaticamente)
    return {"messages": [response]}

builder = StateGraph(MessagesState)  # ← MessagesState aqui
builder.add_node("analyze", analyze_logs)
builder.add_edge(START, "analyze")
builder.add_edge("analyze", END)

checkpointer = InMemorySaver()
graph = builder.compile(checkpointer=checkpointer)

config = {"configurable": {"thread_id": "sessao-teste"}}

# Teste
result1 = graph.invoke(
    {"messages": [HumanMessage(content=SYSTEM_PROMPT)]},
    config
)
print("Resposta 1:", result1["messages"][-1].content)


result2 = graph.invoke(
    {"messages": [HumanMessage(content=f"Analyze this log :\n{data}?")]},
    config)

print(f"Resposta:", result2["messages"][-1].content)

# Verifica estado
estado = graph.get_state(config)
print(f"\nTotal de mensagens: {len(estado.values['messages'])}")
for msg in estado.values['messages']:
    print(f"  [{msg.type}] {msg.content[:80]}...")

LLMFactory: Getting model for provider 'ollama' - metis
Resposta 1: **Event Analysis**

*Event 1:* 
A user downloaded a file named "invoice_12345.pdf" from an external source. The file size is 2MB and the MD5 hash is `abcdef0123456789`.

*Event 2:*
The same file, "invoice_12345.pdf", was executed by a user on their workstation.

**MITRE ATT&CK Mapping**
- **Tactics:** 
- **Techniques:** 
- **Technique IDs:**
LLMFactory: Getting model for provider 'ollama' - metis
Resposta: 

Total de mensagens: 4
  [human] 
You are a cybersecurity analyst specialized in log analysis and MITRE ATT&CK ma...
  [ai] **Event Analysis**

*Event 1:* 
A user downloaded a file named "invoice_12345.pd...
  [human] Analyze this log :
type=SYSCALL msg=audit(1604994496.155:92733): arch=c000003e s...
  [ai] ...


In [8]:
result1 = graph.invoke(
    {"messages": [HumanMessage(content='Provide a final consolidated analysis of all logs and mapping for Mitre ATT&CK')]},
    config
)
print("Resposta 1:", result1["messages"][-1].content)

LLMFactory: Getting model for provider 'ollama' - metis
Resposta 1: **Event Analysis**

*Event 1:* A user downloaded a file named "invoice_12345.pdf" from an external source. The file size is 2MB and the MD5 hash is `abcdef0123456789`.
- **Potential Risk:** The file was downloaded from an external source, which could be a vector for malware or phishing attempts.

*Event 2:* The same file, "invoice_12345.pdf", was executed by a user on their workstation.
- **Observed Behavior:** Execution of the file post-download indicates that the user opened it. This behavior is typical after downloading documents like PDFs.

**MITRE ATT&CK Mapping**
- **Tactics:** 
  - Initial Access
  - Execution
- **Techniques:**
  - Phishing (via external download)
  - Execution of a downloaded file
- **Technique IDs:**
  - T1566: Phishing
  - T1204: User Execution

The initial access could have been through phishing, as the user downloaded the file from an external source. The execution technique is evident by t